In [1]:
import torch
from pathlib import Path
import soundfile as sf
from encodec import EncodecModel
from tts.tokenizer import Tokenizer

ROOT_DIR = Path("..").resolve()
DATA_DIR = ROOT_DIR / "data"

data = torch.load(DATA_DIR / "LJ001-0001.pt")
print("Keys:", data.keys())

audio_tokens = data['audio_tokens']
print(f"\nAudio Shape: {audio_tokens.shape}")
print(f"First 20 tokens: {audio_tokens[:20].tolist()}")

phonemes = data['phonemes']
print(f"\nPhoneme Shape: {phonemes.shape}")
print(f"First 20 phonemes: {phonemes.tolist()}")
print(f"\nText: {data['text']}")

Keys: dict_keys(['audio_tokens', 'phonemes', 'text', 'file_id'])

Audio Shape: torch.Size([725])
First 20 tokens: [780, 194, 887, 887, 252, 418, 57, 939, 120, 994, 57, 928, 3, 3, 871, 25, 887, 904, 66, 744]

Phoneme Shape: torch.Size([159])
First 20 phonemes: [1051, 1072, 1079, 1071, 1049, 1054, 1071, 1062, 1031, 1026, 1071, 1049, 1061, 1071, 1026, 1079, 1050, 1075, 1049, 1047, 1044, 1026, 1053, 1079, 1068, 1049, 1053, 1026, 1057, 1071, 1061, 1026, 1057, 1080, 1071, 1054, 1074, 1026, 1057, 1044, 1081, 1026, 1064, 1081, 1072, 1026, 1060, 1054, 1026, 1051, 1072, 1079, 1068, 1059, 1066, 1049, 1054, 1026, 1046, 1066, 1049, 1053, 1079, 1069, 1081, 1049, 1040, 1031, 1026, 1040, 1079, 1071, 1042, 1067, 1059, 1026, 1042, 1072, 1076, 1048, 1026, 1048, 1079, 1050, 1075, 1053, 1054, 1026, 1071, 1042, 1026, 1049, 1080, 1064, 1081, 1054, 1026, 1042, 1072, 1076, 1048, 1026, 1079, 1065, 1081, 1047, 1026, 1061, 1071, 1026, 1079, 1064, 1081, 1072, 1054, 1053, 1026, 1060, 1049, 1040, 1026, 1046, 1072, 1

In [2]:
tok = Tokenizer()

In [3]:
for tok_id in phonemes.tolist():
    print(tok.id_to_token[tok_id], end="")

pɹˈɪntɪŋ, ɪnðɪ ˈoʊnli sˈɛns wɪð wˌɪtʃ wiː ɑːɹ æt pɹˈɛzənt kənsˈɜːnd, dˈɪfɚz fɹʌm mˈoʊst ɪf nˌɑːt fɹʌm ˈɔːl ðɪ ˈɑːɹts ænd kɹˈæfts ɹˌɛpɹᵻzˈɛntᵻd ɪnðɪ ɛksɪbˈɪʃən<EOS>

In [23]:
codes = audio_tokens.unsqueeze(0).unsqueeze(0)
print("Loading EnCodec...")
model = EncodecModel.encodec_model_24khz()
model.set_target_bandwidth(6.0)
model.eval()


print("Decoding...")
with torch.no_grad():
    decoded_frames = model.decode([(codes, None)])


audio_out = decoded_frames.squeeze().cpu().numpy()
output_path = "reconstructed/debug_reconstruction.wav"
sf.write(output_path, audio_out, 24000)
print(f"Saved to {output_path}")

Loading EnCodec...
Decoding...


/Users/janek/dev/uwr/tts/.venv/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Saved to reconstructed/debug_reconstruction.wav
